In [3]:
!pip install langchain langchain-google-genai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 63.8 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.14.2
    Uninstalling langsmith-0.14.2:
      Successfully uninstalled langsmith-0.14.2
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langchain-core-1.6.6:
      Successfully uninstalled langchain-core-1.

In [4]:
import os                                                                  #التعامل مع نظام التشغيل و ال environment varaibles
import json
from langchain_core.prompts import PromptTemplate                          #عمل برومبت منظم
from langchain_core.output_parsers import PydanticOutputParser             #تتأكد إن ال output ماشي مع ال schema  اللي حاطينها
from pydantic import Field , BaseModel                                     #BaseModel: كلاس بنبني عليها الـSchema بتاعتنا
                                                                           #Field: بنستخدمها عشان نحدد وصفًا أو شروطًا للبيانات
from typing import Optional
from langchain_google_genai import ChatGoogleGenerativeAI                  #دي اللي هتخلينا نتعامل مع Gemini من خلال Google API
from google.colab import userdata

In [5]:
class Command(BaseModel):                                                  #Command: اسم الكلاس اللي بيمثل الأمر المطلوب من الـOS
    action: str = Field(
        description="The action to perform"
    )
    target: Optional[str] = Field(
        default=None,
        description="The target of the action"
    )
    name: Optional[str] = Field(
        default=None,
        description="The name of the item"
    )
    path: Optional[str] = Field(
        default=None,
        description="The file or folder path"
    )
    url: Optional[str] = Field(
        default=None,
        description="The URL to open")
    value: Optional[int] = Field(
        default=None,
        description="Numeric value for the action"
    )

class CommandList(BaseModel):
    commands: list[Command]


In [6]:
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
  try:
        api_key = userdata.get("GOOGLE_API_KEY")
  except (ImportError, Exception):
        api_key = None

if not api_key:
    raise ValueError("API key is missing!")                               #os.getenv() بتقرأ قيمة متخزنة في Environment Variables.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0,
    google_api_key=api_key)


parser = PydanticOutputParser(                                              #يحول رد المودل بحيث يبقى مطابق لل الإسكيما
pydantic_object=CommandList)

prompt = PromptTemplate(                                                    #الـ Prompt هو التعليمات اللي هنقدمها للموديل
    template=(
        "Understand the user's operating system command "
        "and extract the required structured information.\n"
        "If a field is missing, return null for it.\n\n"
        "{format_instructions}\n\n"
        "User command:\n{text}"
    ),
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()             #parser.get_format_instructions(): بتجيب تعليمات التنسيق اللي الـ Parser بيولّدها بناءً على الـ Schema.
    },
)

In [7]:
def extract_information(text: str):
    chain = prompt | llm | parser                                         #بنوصلهم ببعض بالترتيب

    try:
        output: CommandList = chain.invoke({"text": text})                    #chain.invoke() بتشغّل السلسلة وبتبعت البيانات للمراحل اللي جهّزناها.
        print(output.model_dump_json(indent=2))                           #model_dump_json() بتحوّل كائن Pydantic إلى نص بصيغة JSON
                                                                          #indent=2 بتنظّم شكل النتيجة بمسافات عشان تبقى سهلة القراءة
        return output

    except Exception as e:
        print(f"Error: {e}")
        return None



In [ ]:
print("================ TEST CASE 1 ================")

result = extract_information(
    "إفتحلي التيكتوك بعدين ادخل ع جهات الإتصال هاتلي رقم منال السيد"
)



================ TEST CASE 1 ================


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
  "commands": [
    {
      "action": "open",
      "target": "app",
      "name": "TikTok",
      "path": null,
      "url": null
    },
    {
      "action": "open",
      "target": "app",
      "name": "Contacts",
      "path": null,
      "url": null
    },
    {
      "action": "get",
      "target": "phone number",
      "name": "منال السيد",
      "path": null,
      "url": null
    }
  ]
}


In [ ]:
result = extract_information(
    "Create a file named notes.txt in Documents"
)


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Error: Invalid json output: Understood. Please provide the user command you would like me to convert into structured actions, along with the required schema.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [ ]:
result = extract_information(
 " إفتحلي الفيسبوك"
)


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
  "commands": [
    {
      "action": "open",
      "target": "facebook",
      "name": null,
      "path": null,
      "url": "https://www.facebook.com"
    }
  ]
}


In [8]:
result = extract_information("افتح TikTok وعلي الصوت لـ 70%")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{
  "commands": [
    {
      "action": "open",
      "target": "app",
      "name": "TikTok",
      "path": null,
      "url": null,
      "value": null
    },
    {
      "action": "set_volume",
      "target": "system",
      "name": null,
      "path": null,
      "url": null,
      "value": 70
    }
  ]
}
